# 🤖 Joana — AI Chatbot
Run each cell **in order**. Training takes ~1 minute.

## 1. Install dependencies

In [1]:
!pip install colorama --quiet

## 2. Create intents.json

In [2]:
import json

intents = {
  "intents": [
    {"tag": "greeting",
     "patterns": ["Hi", "Hey", "Is anyone there?", "Hello", "Hay"],
     "responses": ["Hello", "Hi", "Hi there"]},
    {"tag": "goodbye",
     "patterns": ["Bye", "See you later", "Goodbye"],
     "responses": ["See you later", "Have a nice day", "Bye! Come back again"]},
    {"tag": "thanks",
     "patterns": ["Thanks", "Thank you", "That's helpful", "Thanks for the help"],
     "responses": ["Happy to help!", "Any time!", "My pleasure", "You're most welcome!"]},
    {"tag": "about",
     "patterns": ["Who are you?", "What are you?", "Who you are?"],
     "responses": ["I'm Joana, your bot assistant", "I'm Joana, an Artificial Intelligent bot"]},
    {"tag": "name",
     "patterns": ["what is your name", "what should I call you", "whats your name?"],
     "responses": ["You can call me Joana.", "I'm Joana!", "Just call me Joana"]},
    {"tag": "help",
     "patterns": ["Could you help me?", "give me a hand please", "Can you help?", "What can you do for me?", "I need a support", "I need a help", "support me please"],
     "responses": ["Tell me how I can assist you", "Tell me your problem to assist you", "Yes Sure, How can I support you"]},
    {"tag": "createaccount",
     "patterns": ["I need to create a new account", "how to open a new account", "I want to create an account", "can you create an account for me"],
     "responses": ["You can easily create a new account from our website", "Just go to our website and follow the guidelines to create a new account"]},
    {"tag": "complaint",
     "patterns": ["have a complaint", "I want to raise a complaint", "there is a complaint about a service"],
     "responses": ["Please provide us your complaint in order to assist you", "Please mention your complaint, we will reach you and sorry for any inconvenience caused"]}
  ]
}

with open("intents.json", "w") as f:
    json.dump(intents, f, indent=2)

print("✅ intents.json created")

✅ intents.json created


## 3. Train the model

In [3]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# ── Hyperparameters ───────────────────────────────────────────
VOCAB_SIZE    = 1000
EMBEDDING_DIM = 16
MAX_LEN       = 20
OOV_TOKEN     = "<OOV>"
EPOCHS        = 500

# ── Load data ─────────────────────────────────────────────────
with open("intents.json") as f:
    data = json.load(f)

training_sentences, training_labels, labels = [], [], []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        training_sentences.append(pattern)
        training_labels.append(intent["tag"])
    if intent["tag"] not in labels:
        labels.append(intent["tag"])

num_classes = len(labels)
print(f"Found {num_classes} intent classes: {labels}")

# ── Encode labels ─────────────────────────────────────────────
lbl_encoder = LabelEncoder()
lbl_encoder.fit(training_labels)
encoded_labels = lbl_encoder.transform(training_labels)

# ── Tokenise & pad ────────────────────────────────────────────
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(training_sentences)
sequences        = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences, truncating="post", maxlen=MAX_LEN)

# ── Build model ───────────────────────────────────────────────
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
    GlobalAveragePooling1D(),
    Dense(16, activation="relu"),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

# ── Train ─────────────────────────────────────────────────────
print("\nTraining … (this may take a minute)")
history = model.fit(padded_sequences, np.array(encoded_labels), epochs=EPOCHS, verbose=0)
final_acc = history.history["accuracy"][-1]
print(f"\n✅ Training complete — final accuracy: {final_acc:.2%}")

Found 8 intent classes: ['greeting', 'goodbye', 'thanks', 'about', 'name', 'help', 'createaccount', 'complaint']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training … (this may take a minute)

✅ Training complete — final accuracy: 100.00%


## 4. Save model artifacts

In [6]:
# Save model
model.save("chat_model.keras")

# Save tokenizer
with open("tokenizer.json", "w") as f:
    json.dump(tokenizer.to_json(), f)

# Save label encoder
with open("label_encoder.json", "w") as f:
    json.dump({"classes": lbl_encoder.classes_.tolist()}, f)

print("✅ Saved: chat_model/  tokenizer.json  label_encoder.json")

✅ Saved: chat_model/  tokenizer.json  label_encoder.json


## 5. Chat with Joana
Type your message and press **Enter**. Type `quit` to stop.

In [9]:
import random
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from colorama import Fore, Style, init

init()

# ── Load artifacts ────────────────────────────────────────────
model_chat = keras.models.load_model("chat_model.keras")


with open("tokenizer.json") as f:
    tok = keras.preprocessing.text.tokenizer_from_json(json.load(f))

with open("label_encoder.json") as f:
    le = LabelEncoder()
    le.classes_ = np.array(json.load(f)["classes"])

with open("intents.json") as f:
    intents_data = json.load(f)

MAX_LEN = 20

print(Fore.YELLOW + "Joana is ready! Type 'quit' to stop.\n" + Style.RESET_ALL)

while True:
    user_input = input( "You: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print(Fore.YELLOW + "Joana: Goodbye! Have a great day! 👋" + Style.RESET_ALL)
        break

    seq     = tok.texts_to_sequences([user_input])
    padded  = pad_sequences(seq, truncating="post", maxlen=MAX_LEN)
    result  = model_chat.predict(padded, verbose=0)
    tag     = le.inverse_transform([np.argmax(result)])[0]

    for intent in intents_data["intents"]:
        if intent["tag"] == tag:
            print(Fore.GREEN + "Joana: " + Style.RESET_ALL + random.choice(intent["responses"]))
            break

Joana is ready! Type 'quit' to stop.

You: Hi
Joana: Hi there
You: Hello
Joana: Hi
You: quit
Joana: Goodbye! Have a great day! 👋
